# 01 · Preprocesamiento y Feature Engineering
**EduPredict** · Samsung Innovation Campus 2025 · Reto 4 · Universidad del Rosario

> **Responsable:** Johan A. Vera Lozano  
> **Objetivo:** Construir el pipeline de preprocesamiento sin data leakage, listo para los dos pipelines del modelo.

---

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.preprocessing import load_and_validate, engineer_features, preprocess, build_preprocessors, text_to_sequences
from src.config import NUMERIC_FEATURES, CLASS_LABELS

UR_RED = "#DA0921"; UR_NAVY = "#242839"; UR_TECH = "#0E6A8C"
print("✅ Importaciones correctas")

## 1 · Carga y feature engineering

In [ ]:
df_raw = load_and_validate("../data/evaluaciones_docentes.csv")
df = engineer_features(df_raw)

print("Features derivados creados:")
print(f"  puntaje_promedio: media de los 3 puntajes -> rango [{df['puntaje_promedio'].min():.2f}, {df['puntaje_promedio'].max():.2f}]")
print(f"  semestre_num:     ordinal 1-8              -> rango [{df['semestre_num'].min()}, {df['semestre_num'].max()}]")
print(f"  asignatura_enc:   label encoding 0-6       -> rango [{df['asignatura_enc'].min()}, {df['asignatura_enc'].max()}]")
print(f"  comentario_clean: texto normalizado         -> ejemplo: '{df['comentario_clean'].iloc[0]}'")
print(f"\nFeatures numéricos para RF ({len(NUMERIC_FEATURES)}):")
for f in NUMERIC_FEATURES:
    print(f"  - {f}")

## 2 · Split estratificado 80/20

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["tendencia_desempeno"]
)

print(f"Train: {len(df_train)} registros")
print(f"Test:  {len(df_test)} registros")
print(f"\nDistribución en train:")
for k, v in Counter(df_train["tendencia_desempeno"]).items():
    print(f"  {k:<12}: {v} ({v/len(df_train)*100:.1f}%)")
print(f"\nDistribución en test:")
for k, v in Counter(df_test["tendencia_desempeno"]).items():
    print(f"  {k:<12}: {v} ({v/len(df_test)*100:.1f}%)")
print("\n✅ Split estratificado - proporciones preservadas en ambos sets")

## 3 · Preprocesadores (ajustados solo con train - sin data leakage)

In [ ]:
tokenizer, scaler, label_encoder = build_preprocessors(df_train)

print(f"Tokenizer - vocabulario: {len(tokenizer.word_index)} palabras únicas")
print(f"  Top 10 tokens:")
top_tokens = sorted(tokenizer.word_index.items(), key=lambda x: x[1])[:10]
for word, idx in top_tokens:
    print(f"    {idx:3}: '{word}'")

print(f"\nScaler - media de cada feature (debe ser ~ valores del train):")
for feat, mean, std in zip(NUMERIC_FEATURES, scaler.mean_, scaler.scale_):
    print(f"  {feat:<28} μ={mean:.3f}  σ={std:.3f}")

print(f"\nLabel Encoder - clases en orden:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i}: {cls}")

## 4 · Transformar texto a secuencias

In [ ]:
X_text_train = text_to_sequences(df_train["comentario_clean"].tolist(), tokenizer)
X_text_test  = text_to_sequences(df_test["comentario_clean"].tolist(), tokenizer)

print(f"X_text_train shape: {X_text_train.shape}  (n_samples × max_len=15)")
print(f"X_text_test  shape: {X_text_test.shape}")
print(f"\nEjemplo - comentario original:  '{df_train['comentario'].iloc[0]}'")
print(f"Comentario limpio:              '{df_train['comentario_clean'].iloc[0]}'")
print(f"Secuencia padded:               {X_text_train[0]}")

## 5 · Transformar features numéricos

In [ ]:
from sklearn.preprocessing import StandardScaler

X_num_train = scaler.transform(df_train[NUMERIC_FEATURES].values)
X_num_test  = scaler.transform(df_test[NUMERIC_FEATURES].values)

print(f"X_num_train shape: {X_num_train.shape}")
print(f"X_num_test  shape: {X_num_test.shape}")
print(f"\nMedia por feature (debe ser ~ 0 en train):")
for feat, mean in zip(NUMERIC_FEATURES, X_num_train.mean(axis=0)):
    print(f"  {feat:<28} {mean:+.4f}")

## 6 · Pipeline completo con `preprocess()`

In [ ]:
# Función que hace todo lo anterior de una sola vez
data = preprocess("../data/evaluaciones_docentes.csv")

print("Shapes del objeto ProcessedData:")
print(f"  X_text_train:  {data.X_text_train.shape}")
print(f"  X_text_test:   {data.X_text_test.shape}")
print(f"  X_num_train:   {data.X_num_train.shape}")
print(f"  X_num_test:    {data.X_num_test.shape}")
print(f"  y_train:       {data.y_train.shape}  - clases: {dict(zip(*np.unique(data.y_train, return_counts=True)))}")
print(f"  y_test:        {data.y_test.shape}   - clases: {dict(zip(*np.unique(data.y_test, return_counts=True)))}")
print("\n✅ Preprocesamiento completado - pasar a 02_cnn1d.ipynb y 03_random_forest.ipynb")